In [1]:
# Customer Churn Prediction

In [2]:
# Objective
#Predict whether a bank customer is likely to churn, and build a model that supports business decision-making by minimizing high-cost errors.

In [3]:
## 🎯 Project Philosophy

### 1. The Core Problem
#The goal is to predict **Customer Churn**. 
#Identifying at-risk customers early allows the bank to intervene with loyalty programs before the customer officially leaves.


### 2. Why Random Forest?
#I chose Random Forest for its **Robustness** and **Interpretability**. 
#It allows us to see which features (like Balance or Age) most heavily influence a customer's decision to stay or leave.


### 3. Strategic Evaluation (Recall vs. Precision)
#In bank churn, a **False Negative** (predicting stay, but they leave) 
#is more expensive than a **False Positive** (predicting leave, but they stay). 
#Therefore, I tuned the model's threshold to 0.2 to maximize **Recall**, ensuring we capture the maximum number of potential churners.

In [4]:
import pandas as pd

# Loading the raw dataset
df = pd.read_csv('Churn_Modelling.csv')

# Visualizing the first 5 rows to understand the feature structure
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
# Dropping irrelevant columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [6]:
#Data Transformation &  Categorical Encoding
#To prepare the dataset for the model, I performed the following encoding techniques:

#Label Encoding: 
#Applied to the Gender feature to convert binary categories (Male/Female) into a machine-readable numerical format (0/1).


#One-Hot Encoding (OHE): 
#Implemented via pd.get_dummies for the Geography feature. 
#This prevents the model from assuming any ordinal relationship or ranking between different countries.

#Avoiding the Dummy Variable Trap:
#Utilized drop_first=True to eliminate multi-collinearity by dropping one redundant column, ensuring the mathematical stability of the model.

In [7]:
from sklearn.preprocessing import  LabelEncoder
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = pd.get_dummies(df, columns = ['Geography'] , drop_first = True)

In [8]:
## feature engineering


#"Calculated the Balance-to-Salary ratio to capture financial health. While absolute balance is static,
#its relation to estimated salary acts as a proxy for customer stickiness and liquidity."
df['BalanceSalaryRatio'] = df['Balance'] / (df['EstimatedSalary'] + 1) 


#"I implemented an Interaction Feature by combining NumOfProducts and IsActiveMember. 
#Data analysis reveals a strong synergy between these two variables;
#an active member holding multiple products demonstrates the highest level of customer engagement, 
#serving as a potent predictor of retention (non-churn)."
df['Active_Product_Score'] = df['IsActiveMember'] * df['NumOfProducts']

In [9]:
#Feature & Target Selection
#In this step, I separated the dataset into independent variables (features) and the dependent variable (target):

#Feature Matrix (X):
#Includes all customer attributes (e.g., Credit Score, Age, Tenure, Balance, and engineered features) after dropping the Exited column.


#Target Vector (y):
#Contains the Exited column, which indicates whether a customer stayed with the bank (0) or left (1).

In [10]:
x = df.drop('Exited' , axis = 1)
y = df['Exited']

In [11]:
#Dataset Splitting
#To ensure the model generalizes well to unseen data, I partitioned the dataset into training and testing sets:

#Training Set (75%): Used to train the model and adjust its internal weights.

#Testing Set (25%): Reserved as a completely unseen dataset to evaluate the model's final performance and accuracy.

#Reproducibility: Set random_state = 42 to ensure the data split is consistent across different runs, making the results verifiable and reproducible.

In [12]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(
    x,y,
    test_size = 0.25 ,
    random_state = 42
)

In [13]:
#Model Selection & Training
#I selected the Random Forest Classifier as the primary model for this classification task 
#due to its robustness and ability to handle non-linear relationships.

#Algorithm: Random Forest (An ensemble of decision trees).

#Hyperparameters:

#n_estimators = 100: Built a forest of 100 decision trees to ensure stable and reliable predictions.


#max_depth = 6: Limited the depth of the trees to prevent Overfitting, 
#ensuring the model learns general patterns rather than memorizing the training noise.


#Training Process: The model was fitted on the x_train and y_train sets, 
#allowing it to learn the decision boundaries between customers who stay and those who churn.

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 6,
    random_state = 42
)

rf_model.fit(x_train , y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
#Model Evaluation & Threshold Optimization
#Rather than relying on the default classification threshold (0.5), 
#I optimized the model's decision-making process to align with the business objective of minimizing customer loss.

#Probability Estimation: 
#Used predict_proba to retrieve the raw likelihood of churn for each customer.

#Custom Threshold (0.2): 
#Lowered the decision threshold to 0.2. This strategic move prioritizes Recall over Precision, 
#ensuring that the bank identifies as many potential churners as possible, even at the cost of more false alarms.

#Performance Metrics:

#Confusion Matrix: To visualize True Positives (correctly identified churners) vs. False Negatives (missed churners).

#Classification Report: To evaluate the trade-off between Precision, Recall, and the F1-Score.

In [16]:
from sklearn.metrics import confusion_matrix , classification_report 
proba = rf_model.predict_proba(x_test)[:, 1]

threshold = 0.2
y_pred_rf = (proba >= threshold).astype(int)

print(confusion_matrix(y_test , y_pred_rf))
print(classification_report(y_test,y_pred_rf))

[[1560  443]
 [ 108  389]]
              precision    recall  f1-score   support

           0       0.94      0.78      0.85      2003
           1       0.47      0.78      0.59       497

    accuracy                           0.78      2500
   macro avg       0.70      0.78      0.72      2500
weighted avg       0.84      0.78      0.80      2500



In [17]:
## Business Insights

#- Customers with low activity and high balances showed higher churn risk.
#- Recall optimization was prioritized to reduce missed churned customers.
#- The model can be used to support targeted retention strategies.